In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NB_04 — Gold Feature Store
# MAGIC **Reads `xscore.silver.user_features` → builds final model-ready feature table → writes to `xscore.gold`**
# MAGIC
# MAGIC What this notebook does:
# MAGIC - Reads the joined Silver feature table
# MAGIC - Defines the exact feature contract (which columns feed the model)
# MAGIC - Normalises skewed features (log-scale income, cap outliers)
# MAGIC - Writes `xscore.gold.credit_feature_store` with CDF enabled
# MAGIC - Saves the feature contract as JSON (MLflow lineage)
# MAGIC - Writes an empty `xscore.gold.credit_scores` table (populated by NB_05)
# MAGIC
# MAGIC **Depends on:** NB_03 complete
# MAGIC **Runtime:** ~3 minutes
# MAGIC **Next:** NB_05_model_training

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 1 — Setup

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import *
import json

spark.sql("USE CATALOG xscore")
spark.conf.set("spark.sql.shuffle.partitions", "8")

print("✓ Setup complete")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 2 — Read Silver features

# COMMAND ----------

silver = spark.table("xscore.silver.user_features")

print(f"Silver user_features: {silver.count():,} rows × {len(silver.columns)} columns")
print(f"\nColumn list:")
for i, c in enumerate(silver.columns):
    print(f"  {i+1:>2}. {c}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 3 — Feature engineering: normalise and cap outliers

# COMMAND ----------

# ─────────────────────────────────────────────────────────────
# Why normalise here and not in Silver?
# Silver keeps raw values for auditability.
# Gold has model-ready transformations.
# The model (GBT) is tree-based so it doesn't strictly need
# normalisation, but log-scaling income reduces the effect of
# extreme outliers pulling splits in wrong directions.
# ─────────────────────────────────────────────────────────────

gold = silver.withColumns({

    # Log-scale income: reduces outlier effect
    # A ₹5K vs ₹10K difference matters more than ₹200K vs ₹205K
    "income_log": F.round(F.log1p(F.col("income_monthly")), 4),

    # Cap UPI transaction count at 99th percentile equivalent (~300)
    # Prevents kirana owners with 500+ txns from dominating splits
    "upi_txn_capped": F.least(F.col("upi_txn_per_month"), F.lit(300)),

    # Cap land acres at 10 — large landowners are outliers
    "land_acres_capped": F.least(F.col("land_acres"), F.lit(10.0)),

    # Cap bank vintage at 120 months (10 years) — very old accounts
    "bank_vintage_capped": F.least(F.col("bank_vintage_months"), F.lit(120)),

    # Cap employment months at 120 months
    "employment_capped": F.least(F.col("employment_months"), F.lit(120)),

    # Convert boolean-ish ints to explicit doubles for Spark MLlib
    "itr_filed_d"       : F.col("itr_filed").cast("double"),
    "gst_registered_d"  : F.col("gst_registered").cast("double"),
    "jan_dhan_active_d" : F.col("jan_dhan_active").cast("double"),
    "shg_member_d"      : F.col("shg_member").cast("double"),
    "owns_land_d"       : F.col("owns_land").cast("double"),
    "owns_vehicle_d"    : F.col("owns_vehicle").cast("double"),
    "has_fd_or_rd_d"    : F.col("has_fd_or_rd").cast("double"),
    "svanidhi_repaid_d" : F.col("svanidhi_repaid").cast("double"),
    "fraud_flag_d"      : F.col("fraud_flag").cast("double"),
})

print(f"Gold features: {gold.count():,} rows × {len(gold.columns)} columns")
print("✓ Transformations applied:")
print("  - income_log          (log1p of income_monthly)")
print("  - upi_txn_capped      (capped at 300)")
print("  - land_acres_capped   (capped at 10 acres)")
print("  - bank_vintage_capped (capped at 120 months)")
print("  - employment_capped   (capped at 120 months)")
print("  - boolean flags cast to double for Spark MLlib")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 4 — Define the feature contract

# COMMAND ----------

# ─────────────────────────────────────────────────────────────
# The feature contract is the EXPLICIT list of columns that
# feed the GBT model. No SELECT * ever touches the model.
# This list is saved as JSON and logged to MLflow in NB_05.
# Any change here must be deliberate and versioned.
# ─────────────────────────────────────────────────────────────

FEATURE_COLS = [
    # ── Pillar 1: Bill payment (5 features) ──────────────────
    "p1_bill_ontime_rate",
    "p1_avg_days_late",
    "p1_severe_late_rate",
    "p1_bill_type_diversity",
    "p1_payment_trend",

    # ── Pillar 2: UPI & digital flow (6 features) ────────────
    "p2_upi_txn_per_month",
    "p2_avg_txn_amount",
    "p2_txn_cv",
    "p2_failure_rate",
    "p2_merchant_diversity",

    # ── Pillar 3: Assets & property (5 features) ─────────────
    "owns_land_d",
    "land_acres_capped",
    "owns_vehicle_d",
    "bank_vintage_capped",
    "has_fd_or_rd_d",

    # ── Pillar 4: Income & employment (4 features) ────────────
    "income_log",
    "itr_filed_d",
    "gst_registered_d",
    "employment_capped",

    # ── Pillar 5: Identity & govt signals (5 features) ────────
    "jan_dhan_active_d",
    "shg_member_d",
    "shg_months",
    "dbt_months",
    "svanidhi_repaid_d",

    # ── Pillar 6: Digital stability (3 features) ─────────────
    "sim_tenure_months",
    "location_stability",
    "fraud_flag_d",

    # ── Derived composites (4 features) ──────────────────────
    "derived_income_to_bills_ratio",
    "derived_digital_engagement",
    "derived_formal_economy_score",
    "derived_asset_score",
]

LABEL_COL = "default_label"

print(f"Feature contract v1.0")
print(f"  Total features : {len(FEATURE_COLS)}")
print(f"  Label column   : {LABEL_COL}")
print(f"\nBreakdown by pillar:")
print(f"  Pillar 1 (bill payment)     : 5 features")
print(f"  Pillar 2 (UPI flow)         : 6 features")
print(f"  Pillar 3 (assets)           : 5 features")
print(f"  Pillar 4 (income/employment): 4 features")
print(f"  Pillar 5 (identity/govt)    : 5 features")
print(f"  Pillar 6 (digital stability): 3 features")
print(f"  Derived composites          : 4 features")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 5 — Write Gold credit feature store

# COMMAND ----------

# Select only what we need: identity cols + features + label
gold_feature_store = gold.select(
    "user_id",
    "segment",
    "state",
    "default_probability",   # keep for analysis, not fed to model
    *FEATURE_COLS,
    LABEL_COL
).fillna(0.0)

(gold_feature_store.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .partitionBy("segment")
    .saveAsTable("xscore.gold.credit_feature_store"))

count = gold_feature_store.count()
print(f"✓ xscore.gold.credit_feature_store")
print(f"  {count:,} rows × {len(gold_feature_store.columns)} columns")
print(f"  CDF enabled")
print(f"  Partitioned by segment")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 6 — Save feature contract to Gold volume

# COMMAND ----------

# Save the feature contract as JSON
# NB_05 reads this so training and serving use identical features
contract = {
    "version"     : "1.0",
    "feature_cols": FEATURE_COLS,
    "label_col"   : LABEL_COL,
    "n_features"  : len(FEATURE_COLS),
    "pillars": {
        "pillar_1_bill_payment"    : FEATURE_COLS[0:5],
        "pillar_2_upi_flow"        : FEATURE_COLS[5:11],
        "pillar_3_assets"          : FEATURE_COLS[11:16],
        "pillar_4_income"          : FEATURE_COLS[16:20],
        "pillar_5_identity"        : FEATURE_COLS[20:25],
        "pillar_6_stability"       : FEATURE_COLS[25:28],
        "derived_composites"       : FEATURE_COLS[28:32],
    }
}

contract_json = json.dumps(contract, indent=2)

# Write to a UC Volume so all notebooks can read it
dbutils.fs.put(
    "/Volumes/xscore/bronze/kaggle_raw/feature_contract.json",
    contract_json,
    overwrite=True
)

print("✓ Feature contract saved:")
print(f"  /Volumes/xscore/bronze/kaggle_raw/feature_contract.json")
print(f"\n{contract_json}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 7 — Create empty credit_scores Gold table

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Create the credit_scores table now (empty).
# MAGIC -- NB_05 will MERGE scored results into this table after training.
# MAGIC -- We define the schema explicitly so it's stable across runs.
# MAGIC
# MAGIC CREATE TABLE IF NOT EXISTS xscore.gold.credit_scores (
# MAGIC   user_id             STRING    NOT NULL,
# MAGIC   segment             STRING,
# MAGIC   xscore              INT,
# MAGIC   score_band          STRING,
# MAGIC   default_probability DOUBLE,
# MAGIC   score_timestamp     TIMESTAMP,
# MAGIC   model_version       STRING,
# MAGIC   model_run_id        STRING
# MAGIC )
# MAGIC USING DELTA
# MAGIC TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
# MAGIC COMMENT 'XScore credit scores — one row per user, updated by NB_05 and NB_09';

# COMMAND ----------

print("✓ xscore.gold.credit_scores table created (empty)")
print("  Will be populated by NB_05 after model training")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 8 — Quick feature distribution check

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Verify feature distributions look sensible before training
# MAGIC -- income_log should be ~9-10 for Indian incomes (log of ₹8K-₹30K)
# MAGIC -- p1_bill_ontime_rate should average ~0.72-0.75 across all segments
# MAGIC -- default_label should be ~8-12%
# MAGIC
# MAGIC SELECT
# MAGIC   ROUND(AVG(income_log), 3)                AS avg_income_log,
# MAGIC   ROUND(AVG(p1_bill_ontime_rate), 3)       AS avg_bill_ontime,
# MAGIC   ROUND(AVG(p2_upi_txn_per_month), 1)      AS avg_upi_txn,
# MAGIC   ROUND(AVG(derived_formal_economy_score), 1) AS avg_formal_score,
# MAGIC   ROUND(AVG(derived_asset_score), 1)       AS avg_asset_score,
# MAGIC   ROUND(AVG(default_label), 4)             AS overall_default_rate,
# MAGIC   COUNT(*)                                 AS total_users
# MAGIC FROM xscore.gold.credit_feature_store;

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 9 — Final summary

# COMMAND ----------

print("=" * 60)
print("  NB_04 GOLD FEATURE STORE — COMPLETE")
print("=" * 60)

for tbl in ["credit_feature_store", "credit_scores"]:
    n = spark.table(f"xscore.gold.{tbl}").count()
    print(f"  ✓  gold.{tbl:<25} {n:>10,} rows")

print("=" * 60)
print()
print(f"  Feature contract: {len(FEATURE_COLS)} features across 6 pillars")
print(f"  Saved to: /Volumes/xscore/bronze/kaggle_raw/feature_contract.json")
print()
print("  NEXT: Run NB_05_model_training")
print("  NB_05 trains 3 GBT versions in MLflow,")
print("  registers @Champion, scores all users,")
print("  and writes results to gold.credit_scores.")
print("=" * 60)